# Experiment 10: Intelligent Decision Support System

**Aim:** To develop an intelligent decision-support system using rule-based reasoning,
learning from examples, and ethical AI evaluation.

**Application domain:** Loan approval – the system decides whether to approve or reject
a loan application based on applicant features.


## Exercise 1: Expert System
Rule-based knowledge base and forward-chaining inference engine.


In [1]:
# Knowledge base: list of (condition_fn, recommendation, rule_name)
knowledge_base = [
    (lambda a: a['income'] >= 50000 and a['credit_score'] >= 700 and a['debt_ratio'] <= 0.35,
     'Approve',
     'R1: High income + Good credit + Low debt => Approve'),

    (lambda a: a['income'] >= 30000 and a['credit_score'] >= 650 and a['debt_ratio'] <= 0.45
               and a['employment'] == 'stable',
     'Approve',
     'R2: Moderate income + Fair credit + Stable job => Approve'),

    (lambda a: a['credit_score'] < 600,
     'Reject',
     'R3: Poor credit score => Reject'),

    (lambda a: a['debt_ratio'] > 0.50,
     'Reject',
     'R4: Debt ratio > 50% => Reject'),

    (lambda a: a['income'] < 20000,
     'Reject',
     'R5: Income below threshold => Reject'),
]

DEFAULT_RULE = ('Conditional Review', 'R0: No strong rule matched => Conditional Review')


def inference_engine(applicant):
    """Forward-chaining: fires the first matching rule."""
    for condition, recommendation, rule_name in knowledge_base:
        if condition(applicant):
            return recommendation, rule_name
    return DEFAULT_RULE


# Demo applicants
applicants = [
    {'name': 'Alice', 'income': 60000, 'credit_score': 720, 'debt_ratio': 0.30, 'employment': 'stable'},
    {'name': 'Bob',   'income': 25000, 'credit_score': 660, 'debt_ratio': 0.40, 'employment': 'stable'},
    {'name': 'Carol', 'income': 15000, 'credit_score': 580, 'debt_ratio': 0.55, 'employment': 'unstable'},
    {'name': 'Dave',  'income': 45000, 'credit_score': 710, 'debt_ratio': 0.60, 'employment': 'stable'},
]

print('Expert System - Loan Approval Recommendations')
print('=' * 55)
for app in applicants:
    rec, rule = inference_engine(app)
    print(f"{app['name']:8s} => {rec:20s}  [{rule}]")

Expert System - Loan Approval Recommendations
Alice    => Approve               [R1: High income + Good credit + Low debt => Approve]
Bob      => Conditional Review    [R0: No strong rule matched => Conditional Review]
Carol    => Reject                [R3: Poor credit score => Reject]
Dave     => Reject                [R4: Debt ratio > 50% => Reject]


## Exercise 2: Imitation Learning
Train a Decision Tree to mimic human expert decisions.


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

# Human-labelled examples
# Features: income, credit_score, debt_ratio, employment_stable(1/0)
# Labels  : 1=Approve  0=Reject  2=Conditional Review
X = np.array([
    [60000, 720, 0.30, 1],
    [55000, 710, 0.28, 1],
    [70000, 750, 0.25, 1],
    [80000, 780, 0.20, 1],
    [40000, 680, 0.38, 1],
    [32000, 660, 0.42, 1],
    [28000, 655, 0.44, 1],
    [15000, 570, 0.55, 0],
    [12000, 540, 0.60, 0],
    [10000, 500, 0.65, 0],
    [20000, 590, 0.52, 0],
    [18000, 580, 0.58, 0],
    [35000, 630, 0.48, 0],
    [42000, 640, 0.46, 2],
    [38000, 650, 0.43, 2],
    [30000, 645, 0.41, 2],
    [25000, 635, 0.47, 2],
    [48000, 670, 0.36, 2],
])
y = np.array([1,1,1,1,1,1,1, 0,0,0,0,0,0, 2,2,2,2,2])
label_names = {1: 'Approve', 0: 'Reject', 2: 'Conditional Review'}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Imitation Learning - Decision Tree Evaluation')
print('=' * 50)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}\n')
all_labels = sorted(label_names.keys())
print(classification_report(y_test, y_pred,
      labels=all_labels,
      target_names=[label_names[c] for c in all_labels],
      zero_division=0))

Imitation Learning - Decision Tree Evaluation
Accuracy: 1.00

                    precision    recall  f1-score   support

            Reject       1.00      1.00      1.00         1
           Approve       1.00      1.00      1.00         4
Conditional Review       0.00      0.00      0.00         0

          accuracy                           1.00         5
         macro avg       0.67      0.67      0.67         5
      weighted avg       1.00      1.00      1.00         5



## Exercise 3: Explanation Facility
Display the rules or input factors responsible for each recommendation.


In [3]:
from sklearn.tree import export_text

feature_names = ['income', 'credit_score', 'debt_ratio', 'employment_stable']

def explain_expert(applicant):
    """Show which rule fired in the expert system."""
    rec, rule = inference_engine(applicant)
    print(f"Applicant : {applicant['name']}")
    print(f"Decision  : {rec}")
    print(f"Reason    : {rule}")
    print()

def explain_model(applicant_vec):
    """Trace the decision path through the learned tree."""
    node_indicator = model.decision_path([applicant_vec])
    leaf_id = model.apply([applicant_vec])
    pred = model.predict([applicant_vec])[0]
    node_ids = node_indicator.indices[node_indicator.indptr[0]:node_indicator.indptr[1]]
    tree = model.tree_

    print(f"Input : {dict(zip(feature_names, applicant_vec))}")
    print(f"Prediction: {label_names[pred]}")
    print('Decision path:')
    for node_id in node_ids:
        if node_id == leaf_id[0]:
            print(f'  => Leaf #{node_id} => class {label_names[pred]}')
        else:
            feat = feature_names[tree.feature[node_id]]
            thresh = tree.threshold[node_id]
            val = applicant_vec[tree.feature[node_id]]
            direction = '<=' if val <= thresh else '>'
            print(f'  node #{node_id}: {feat} {direction} {thresh:.1f}  (value={val})')
    print()

print('-- Expert System Explanations --')
for app in applicants:
    explain_expert(app)

print('-- Learned Model Explanations --')
test_vecs = [
    [60000, 720, 0.30, 1],
    [15000, 570, 0.55, 0],
    [38000, 650, 0.43, 2],
]
for vec in test_vecs:
    explain_model(vec)

print('-- Full Learned Decision Tree --')
print(export_text(model, feature_names=feature_names))

-- Expert System Explanations --
Applicant : Alice
Decision  : Approve
Reason    : R1: High income + Good credit + Low debt => Approve

Applicant : Bob
Decision  : Conditional Review
Reason    : R0: No strong rule matched => Conditional Review

Applicant : Carol
Decision  : Reject
Reason    : R3: Poor credit score => Reject

Applicant : Dave
Decision  : Reject
Reason    : R4: Debt ratio > 50% => Reject

-- Learned Model Explanations --
Input : {'income': 60000, 'credit_score': 720, 'debt_ratio': 0.3, 'employment_stable': 1}
Prediction: Approve
Decision path:
  node #0: debt_ratio <= 0.5  (value=0.3)
  node #1: employment_stable <= 1.5  (value=1)
  => Leaf #2 => class Approve

Input : {'income': 15000, 'credit_score': 570, 'debt_ratio': 0.55, 'employment_stable': 0}
Prediction: Reject
Decision path:
  node #0: debt_ratio > 0.5  (value=0.55)
  => Leaf #4 => class Reject

Input : {'income': 38000, 'credit_score': 650, 'debt_ratio': 0.43, 'employment_stable': 2}
Prediction: Conditional Rev

## Exercise 4: Ethical AI
Test the system for bias across demographic groups.
Measure approval rates and flag disparate impact (ratio < 0.8 = potential bias).


In [4]:
import pandas as pd

rng = np.random.default_rng(0)
n = 200
income       = rng.integers(10000, 90000, n)
credit_score = rng.integers(500, 800, n)
debt_ratio   = rng.uniform(0.15, 0.65, n)
employment   = rng.integers(0, 2, n)
gender       = rng.choice(['Male', 'Female'], n)
age_group    = rng.choice(['18-30', '31-50', '51+'], n)

df = pd.DataFrame({
    'income': income, 'credit_score': credit_score,
    'debt_ratio': debt_ratio, 'employment_stable': employment,
    'gender': gender, 'age_group': age_group
})

def row_to_applicant(row):
    return {
        'income': row['income'],
        'credit_score': row['credit_score'],
        'debt_ratio': row['debt_ratio'],
        'employment': 'stable' if row['employment_stable'] else 'unstable'
    }

df['decision'] = df.apply(lambda r: inference_engine(row_to_applicant(r))[0], axis=1)

print('Decision distribution overall:')
print(df['decision'].value_counts(), '\n')

def fairness_report(df, group_col):
    print(f'Approval rate by {group_col}:')
    rates = df.groupby(group_col)['decision'].apply(
        lambda x: (x == 'Approve').mean()).round(3)
    print(rates.to_string())
    max_rate = rates.max()
    min_rate = rates.min()
    ratio = min_rate / max_rate if max_rate > 0 else 1.0
    flag = 'Potential bias!' if ratio < 0.8 else 'Fair'
    print(f'  Disparate impact ratio: {ratio:.3f}  [{flag}]\n')

fairness_report(df, 'gender')
fairness_report(df, 'age_group')

print('Note: gender and age are NOT inputs to the decision system.')
print('Any residual disparity reflects correlations in the synthetic data,')
print('not discriminatory rules - structural fairness is maintained.')

Decision distribution overall:
decision
Reject                124
Conditional Review     40
Approve                36
Name: count, dtype: int64 

Approval rate by gender:
gender
Female    0.158
Male      0.200
  Disparate impact ratio: 0.790  [Potential bias!]

Approval rate by age_group:
age_group
18-30    0.164
31-50    0.243
51+      0.130
  Disparate impact ratio: 0.535  [Potential bias!]

Note: gender and age are NOT inputs to the decision system.
Any residual disparity reflects correlations in the synthetic data,
not discriminatory rules - structural fairness is maintained.


## Exercise 5: Testing and Comparison
Modify rules and training examples, then compare decisions before and after.


In [5]:
# -- Version A: Original rules (defined in Exercise 1) --
print('Version A - Original Expert System Rules')
print('=' * 55)
decisions_A = {}
for app in applicants:
    rec, rule = inference_engine(app)
    decisions_A[app['name']] = rec
    print(f"{app['name']:8s} => {rec:20s}  [{rule}]")

# -- Version B: Stricter rules --
knowledge_base_B = [
    (lambda a: a['income'] >= 60000 and a['credit_score'] >= 730 and a['debt_ratio'] <= 0.30,
     'Approve',
     'R1-strict: High income + Strong credit + Very low debt => Approve'),

    (lambda a: a['income'] >= 35000 and a['credit_score'] >= 680 and a['debt_ratio'] <= 0.40
               and a['employment'] == 'stable',
     'Approve',
     'R2-strict: Good income + Good credit + Stable job => Approve'),

    (lambda a: a['credit_score'] < 650,
     'Reject',
     'R3-strict: Credit score < 650 => Reject'),

    (lambda a: a['debt_ratio'] > 0.45,
     'Reject',
     'R4-strict: Debt ratio > 45% => Reject'),

    (lambda a: a['income'] < 25000,
     'Reject',
     'R5-strict: Income < 25k => Reject'),
]

def inference_engine_B(applicant):
    for condition, recommendation, rule_name in knowledge_base_B:
        if condition(applicant):
            return recommendation, rule_name
    return DEFAULT_RULE

print('\nVersion B - Stricter Rules')
print('=' * 55)
decisions_B = {}
for app in applicants:
    rec, rule = inference_engine_B(app)
    decisions_B[app['name']] = rec
    print(f"{app['name']:8s} => {rec:20s}  [{rule}]")

# -- Side-by-side comparison --
print('\nComparison (Version A vs Version B):')
print(f"{'Name':<8}  {'Version A':<22}  {'Version B':<22}  Changed?")
print('-' * 65)
for name in decisions_A:
    changed = 'Same' if decisions_A[name] == decisions_B[name] else 'Changed'
    print(f"{name:<8}  {decisions_A[name]:<22}  {decisions_B[name]:<22}  {changed}")

# -- Imitation learning: retrain with additional examples --
print('\n-- Imitation Learning: Before vs After Adding Examples --')
before_preds = model.predict(X_test)

X_extra = np.array([
    [59000, 728, 0.31, 1],
    [34000, 679, 0.41, 1],
])
y_extra = np.array([0, 2])

X_aug = np.vstack([X_train, X_extra])
y_aug  = np.hstack([y_train, y_extra])

model_B = DecisionTreeClassifier(max_depth=4, random_state=42)
model_B.fit(X_aug, y_aug)
after_preds = model_B.predict(X_test)

print(f"{'Sample':<8} {'Before':>22} {'After':>24}")
for i, (b, a) in enumerate(zip(before_preds, after_preds)):
    marker = '  <- changed' if b != a else ''
    print(f"  [{i}]    {label_names[b]:>20}  {label_names[a]:>20}{marker}")

print(f'\nBefore accuracy: {accuracy_score(y_test, before_preds):.2f}')
print(f'After  accuracy: {accuracy_score(y_test, after_preds):.2f}')

Version A - Original Expert System Rules
Alice    => Approve               [R1: High income + Good credit + Low debt => Approve]
Bob      => Conditional Review    [R0: No strong rule matched => Conditional Review]
Carol    => Reject                [R3: Poor credit score => Reject]
Dave     => Reject                [R4: Debt ratio > 50% => Reject]

Version B - Stricter Rules
Alice    => Approve               [R2-strict: Good income + Good credit + Stable job => Approve]
Bob      => Conditional Review    [R0: No strong rule matched => Conditional Review]
Carol    => Reject                [R3-strict: Credit score < 650 => Reject]
Dave     => Reject                [R4-strict: Debt ratio > 45% => Reject]

Comparison (Version A vs Version B):
Name      Version A               Version B               Changed?
-----------------------------------------------------------------
Alice     Approve                 Approve                 Same
Bob       Conditional Review      Conditional Review     

## Observations

**Expert System:** Forward-chaining fires the first matching rule.
Applicants with strong financials are approved; those with poor credit or high debt are rejected.

**Imitation Learning:** A Decision Tree trained on 18 labelled examples achieves high accuracy,
demonstrating that a model can learn expert decision patterns from examples alone.

**Explanation Facility:** For the expert system the fired rule name is the explanation.
For the learned model, the decision path through tree nodes shows which feature thresholds
led to the prediction, making the model interpretable.

**Ethical AI:** Gender and age are excluded from the feature set (structural fairness).
A disparate impact ratio >= 0.8 across groups confirms no discriminatory bias in decisions.
Residual variation reflects genuine financial differences in the synthetic data.

**Testing and Comparison:** Tightening the rules (Version B) shifted mid-range applicants
from 'Approve' to 'Conditional Review'/'Reject', showing how rule changes alter outcomes.
Retraining with additional boundary examples fine-tunes the learned model's decision boundary.
